# #18 — Drug-Resistance Prediction from Genotype (Staphylococcus aureus)

**Study `PROJECT_NARRATIVE.md` then `WORKFLOW.md` before touching this notebook.**

~80–90% of this notebook is blank (`____`) for you to fill in from the hint above each cell + the linked docs — **including Step 0's data-fetching queries now.** Real-world bioinformatics work is increasingly API/live-data driven, so writing the actual RQL query strings and turning the response into a clean DataFrame is part of what this project teaches, not something to skip past.

**This is the SECOND real dataset pivot in this project** — see `PROJECT_NARRATIVE.md` Section 7a (CRyPTIC/*M. tuberculosis* → BV-BRC) and Section 7b (*K. pneumoniae* → *S. aureus*, found by directly measuring how many genomes actually have BOTH real phenotype testing AND curated AMR-gene data — it turned out to be too few for *K. pneumoniae*, even for the exact causal gene family).

Species: *Staphylococcus aureus* (BV-BRC taxon_id `1280`). Drugs: **methicillin** (MRSA resistance, driven by the **mecA/mecC** gene cassette) and **erythromycin** (macrolide resistance, driven by the **erm** gene family). Step 2 does NOT pre-filter features to these known genes — it builds a broad feature matrix from every curated AMR gene, and Step 5 checks whether mecA/erm rise to the top on their own.

## Step 0 — Query real AMR phenotype + gene + strain-typing data

**WHAT:** Three live RQL queries against BV-BRC's Data API (`https://www.bv-brc.org/api/{TYPE}/?{QUERY}`):
1. `genome_amr` — per-isolate resistant/susceptible phenotype calls for methicillin + erythromycin.
2. `sp_gene` — ALL curated AMR gene presence calls (CARD-sourced) for the species (not filtered to any drug or gene family — Step 2 needs the full picture).
3. `genome` — per-isolate MLST sequence type (the strain-clustering variable for Step 3).

**Only the generic request/parsing helper (`bvbrc_query`) below is GIVEN** — true reusable boilerplate. Building each actual query string and turning its response into a clean, filtered DataFrame is yours to write in the three WRITE-IT cells that follow, using the real field names/values already verified live (listed below) — you don't need to re-discover these, but you DO need to assemble the query and the pandas logic yourself.

**Real gotchas found live, already verified — use them, don't re-derive them:**
1. RQL syntax: `eq(field,value)`, `select(f1,f2,...)`, `limit(count,start)`. URL-encode field names and values *individually*, not the whole expression, if you ever need a value with special characters.
2. `antibiotic` values are lowercase generic names (`"methicillin"`, not `"Methicillin"`).
3. `evidence` values are really `"Laboratory Method"`/`"Computational Method"` (the doc page names `Phenotype`/`Genotype`/`Literature`/`Comment` — don't trust that).
4. `sp_gene`'s `property` value for AMR genes is really `"Antibiotic Resistance"` (the doc page names `"AMR"`).
5. **Biggest one:** `and(eq(...),eq(...))` / `in(field,(...))` returned **0 rows** in live testing, even with fully correct values. Use a single `eq(taxon_id,...)` in every query and filter everything else (drugs, property value) in pandas afterward, not inside the query string.
6. **Why S. aureus and not K. pneumoniae:** even after fixing 1–5, K. pneumoniae's phenotype-tested and gene-curated genomes barely overlapped (~7–18 genomes). Measuring the same overlap for S. aureus found ~400+ genomes per drug — see `PROJECT_NARRATIVE.md` Section 7b.

In [1]:
import requests
import pandas as pd
import numpy as np
import os
from io import StringIO

os.makedirs("data_py", exist_ok=True)
os.makedirs("results_py", exist_ok=True)

BVBRC_BASE = "https://www.bv-brc.org/api"
SAUREUS_TAXON_ID = 1280
DRUGS = ["methicillin", "erythromycin"]  # BV-BRC stores lowercase generic drug names -- verified live
AMR_PROPERTY_VALUE = "Antibiotic Resistance"  # sp_gene's real live value -- doc page wrongly says "AMR"

def bvbrc_query(data_type, rql_query, fmt="json"):
    """GIVEN -- generic BV-BRC RQL request/parse helper, reused by all three queries below.
    Returns a list of dicts for json, a DataFrame for csv/tsv."""
    accept = {"json": "application/json", "csv": "text/csv", "tsv": "text/tsv"}[fmt]
    url = f"{BVBRC_BASE}/{data_type}/?{rql_query}"
    resp = requests.get(url, headers={"Accept": accept}, timeout=120)
    resp.raise_for_status()
    if fmt == "json":
        return resp.json()
    sep = "," if fmt == "csv" else "\t"
    return pd.read_csv(StringIO(resp.text), sep=sep)


### Step 0a — Query 1: `genome_amr` phenotype calls  ⟵ WRITE IT

**WHAT/WHY/HOW:**
1. Build the RQL query string: a single `eq(taxon_id,SAUREUS_TAXON_ID)` (remember gotcha #5 — no `and()`/`in()`), `select()` the columns you'll need downstream (`genome_id`, `genome_name`, `antibiotic`, `resistant_phenotype`, `evidence`, `computational_method`, `computational_method_performance`), and `limit(50000)` so you get everything for this taxon in one call.
2. Call `bvbrc_query("genome_amr", your_query, fmt="json")` and wrap the result in a DataFrame.
3. Filter to `DRUGS` (methicillin + erythromycin) using `.isin()` — this is the pandas filtering step that replaces the broken `in()` RQL operator.
4. Save to `data_py/genome_amr.csv` (feeds the R twin).

**Docs/functions to use:** f-string query construction, `pd.DataFrame(records)`, boolean filtering with `.isin()`, `df.to_csv()`.

In [2]:
amr_query = f"eq(taxon_id,{SAUREUS_TAXON_ID})&select(genome_id,genome_name,antibiotic,resistant_phenotype,evidence,computational_method,computational_method_performance)&limit(50000)"  # f"eq(taxon_id,{SAUREUS_TAXON_ID})&select(genome_id,genome_name,antibiotic,resistant_phenotype,evidence,computational_method,computational_method_performance)&limit(50000)"
genome_amr_records = bvbrc_query("genome_amr", amr_query, fmt="json")  # bvbrc_query("genome_amr", amr_query, fmt="json")
genome_amr_df = pd.DataFrame(genome_amr_records)  # pd.DataFrame(genome_amr_records)
print(f"genome_amr (all S. aureus, all drugs): {len(genome_amr_df)} rows")

if len(genome_amr_df) > 0:
    genome_amr_df = genome_amr_df[genome_amr_df["antibiotic"].isin(DRUGS)].copy()  # genome_amr_df[genome_amr_df["antibiotic"].isin(DRUGS)].copy()

genome_amr_df.to_csv("data_py/genome_amr.csv", index=False)  # genome_amr_df.to_csv("data_py/genome_amr.csv", index=False)
print(f"genome_amr (methicillin + erythromycin only): {len(genome_amr_df)} rows")
print(genome_amr_df.head())


genome_amr (all S. aureus, all drugs): 25000 rows
genome_amr (methicillin + erythromycin only): 5697 rows
      antibiotic resistant_phenotype              evidence   genome_id  \
6    methicillin           Resistant  Computational Method  1280.38521   
8   erythromycin         Susceptible  Computational Method  1280.26402   
14   methicillin           Resistant  Computational Method  1280.46909   
21  erythromycin           Resistant  Computational Method  1280.38010   
25   methicillin         Susceptible  Computational Method  1280.36789   

                             genome_name computational_method  \
6             Staphylococcus aureus WH-7  AdaBoost Classifier   
8    Staphylococcus aureus strain 23-977  AdaBoost Classifier   
14          Staphylococcus aureus SA-675  AdaBoost Classifier   
21            Staphylococcus aureus 0927  AdaBoost Classifier   
25  Staphylococcus aureus strain 37920-7  AdaBoost Classifier   

             computational_method_performance  
6   Accura

### Step 0b — Query 2: `sp_gene` curated AMR genes  ⟵ WRITE IT

**WHAT/WHY/HOW:** same pattern as Query 1, but do NOT filter to any drug or gene family here — Step 2 needs every curated AMR gene, since the feature matrix is built broad on purpose (see `PROJECT_NARRATIVE.md` Section 7b for why).
1. Build the query: `eq(taxon_id,SAUREUS_TAXON_ID)`, `select(genome_id,gene,property,antibiotics,antibiotics_class,classification,function,source)`, `limit(50000)`.
2. Fetch, wrap in a DataFrame, print the distinct `property` values you actually see (a real sanity check, not busywork — confirms `AMR_PROPERTY_VALUE` still matches live data).
3. Filter rows to `property == AMR_PROPERTY_VALUE` — keep ALL genes that pass this filter, don't narrow further.
4. Save to `data_py/sp_gene.csv`.

**Docs/functions to use:** same as Query 1, plus `.unique()` for the sanity-check print.

In [3]:
sp_gene_query = f"eq(taxon_id,{SAUREUS_TAXON_ID})&select(genome_id,gene,property,antibiotics,antibiotics_class,classification,function,source)&limit(50000)"  # f"eq(taxon_id,{SAUREUS_TAXON_ID})&select(genome_id,gene,property,antibiotics,antibiotics_class,classification,function,source)&limit(50000)"
sp_gene_records = bvbrc_query("sp_gene", sp_gene_query, fmt="json")  # bvbrc_query("sp_gene", sp_gene_query, fmt="json")
sp_gene_all_df = pd.DataFrame(sp_gene_records)  # pd.DataFrame(sp_gene_records)
print(f"sp_gene (all S. aureus, all properties): {len(sp_gene_all_df)} rows")

if len(sp_gene_all_df) > 0 and "property" in sp_gene_all_df.columns:
    print("Real property values found:", sorted(sp_gene_all_df["property"].dropna().unique()))  # sorted(sp_gene_all_df["property"].dropna().unique())

sp_gene_df = sp_gene_all_df[sp_gene_all_df["property"] == AMR_PROPERTY_VALUE].copy() if len(sp_gene_all_df) > 0 else sp_gene_all_df  # sp_gene_all_df[sp_gene_all_df["property"] == AMR_PROPERTY_VALUE].copy() if len(sp_gene_all_df) > 0 else sp_gene_all_df

sp_gene_df.to_csv("data_py/sp_gene.csv", index=False)  # sp_gene_df.to_csv("data_py/sp_gene.csv", index=False)
print(f"sp_gene ({AMR_PROPERTY_VALUE} only, ALL genes kept -- not filtered to any family): {len(sp_gene_df)} rows")
print("Distinct genes:", sp_gene_df["gene"].nunique() if len(sp_gene_df) else 0)  # sp_gene_df["gene"].nunique() if len(sp_gene_df) else 0
print(sp_gene_df.head())


sp_gene (all S. aureus, all properties): 25000 rows
Real property values found: ['Antibiotic Resistance', 'Drug Target', 'Human Homolog', 'Transporter', 'Virulence Factor']
sp_gene (Antibiotic Resistance only, ALL genes kept -- not filtered to any family): 5597 rows
Distinct genes: 189
           antibiotics_class  \
4                              
7       Isoniazid, Triclosan   
13                       NaN   
20  Glycopeptide antibiotics   
23                       NaN   

                                             function   genome_id  \
4   MULTISPECIES: trimethoprim-resistant dihydrofo...  1280.11042   
7   3-oxoacyl-[acyl-carrier-protein] synthase, KAS...  1280.25729   
13  MULTISPECIES: beta-lactam sensor/signal transd...  1280.30261   
20  Teicoplanin resistance transporter, TcaB famil...  1280.21086   
23                                                     1280.24196   

                 property  source  \
4   Antibiotic Resistance   NDARO   
7   Antibiotic Resistance  PATR

### Step 0c — Query 3: `genome` metadata (MLST)  ⟵ WRITE IT

**WHAT/WHY/HOW:** simplest of the three — a single `eq(taxon_id,SAUREUS_TAXON_ID)`, `select(genome_id,genome_name,mlst,strain)`, `limit(50000)`. No filtering needed afterward; you just need every genome's MLST value for Step 3's structure-aware split. Print the fraction of genomes with a non-null `mlst` as a sanity check (you should see something well above 0%, similar to earlier checks in this project).

**Docs/functions to use:** same query-building pattern as Query 1/2; `.notna().mean()` for the non-null fraction.

In [4]:
genome_query = f"eq(taxon_id,{SAUREUS_TAXON_ID})&select(genome_id,genome_name,mlst,strain)&limit(50000)" # f"eq(taxon_id,{SAUREUS_TAXON_ID})&select(genome_id,genome_name,mlst,strain)&limit(50000)"
genome_records = bvbrc_query("genome", genome_query, fmt="json")  # bvbrc_query("genome", genome_query, fmt="json")
genome_meta_df = pd.DataFrame(genome_records)  # pd.DataFrame(genome_records)
genome_meta_df.to_csv("data_py/genome_metadata.csv", index=False)  # genome_meta_df.to_csv("data_py/genome_metadata.csv", index=False)
print(f"genome metadata: {len(genome_meta_df)} rows")
print(genome_meta_df.head())
print("\nMLST non-null fraction:", genome_meta_df["mlst"].notna().mean() if "mlst" in genome_meta_df.columns else "mlst column missing"
)  # genome_meta_df["mlst"].notna().mean() if "mlst" in genome_meta_df.columns else "mlst column missing"


genome metadata: 25000 rows
              strain             mlst   genome_id  \
0         19KS01SA08   MLST.saureus.1  1280.62799   
1             2014.N  MLST.saureus.30  1280.48751   
2  2014.N Survivor A  MLST.saureus.30  1280.48752   
3  2014.N Survivor B  MLST.saureus.30  1280.48753   
4              43187  MLST.saureus.45  1280.53344   

                               genome_name  
0         Staphylococcus aureus 19KS01SA08  
1             Staphylococcus aureus 2014.N  
2  Staphylococcus aureus 2014.N Survivor A  
3  Staphylococcus aureus 2014.N Survivor B  
4              Staphylococcus aureus 43187  

MLST non-null fraction: 0.94224


## Step 1 — Build the binary phenotype labels  ⟵ WRITE IT

**BIOLOGY:** `resistant_phenotype` is a real lab or literature-derived call: `Resistant`, `Susceptible`, `Intermediate`, or `Non-susceptible`. Keep only the confident binary calls (drop `Intermediate`/`Non-susceptible` — the same censored-value discipline used throughout this portfolio).

**WHAT/WHY/HOW:**
1. A genome can have multiple `genome_amr` rows per drug from different evidence sources. The REAL live `evidence` values are `"Laboratory Method"` and `"Computational Method"` (confirmed by querying real records — the doc page's `Phenotype`/`Genotype`/`Literature`/`Comment` doesn't match). Use **only `evidence == "Laboratory Method"`** rows as your real training/test labels — actual lab-measured (disk diffusion / broth dilution / MIC) results. Set aside `evidence == "Computational Method"` rows separately — BV-BRC's own AdaBoost-classifier calls, each carrying a `computational_method_performance` string (e.g. `"Accuracy:0.931, F1 score:0.935, AUC:0.950"`) — you'll use these in Step 4b as a ready-made baseline.
2. Filter to each drug; map `Resistant` → 1, `Susceptible` → 0; drop everything else.
3. If a genome has more than one `Laboratory Method` row for the same drug (duplicate/conflicting), decide and document a rule (e.g. drop conflicting duplicates rather than silently picking one).

**Docs/functions to use:** boolean filtering `df[df['col'] == value]`, `.map({...})` for the binary recode, `.duplicated(subset=..., keep=False)` to find and inspect conflicting duplicates before deciding how to handle them.

In [5]:
def build_labels(genome_amr_df, drug, evidence_type="Laboratory Method"):
    """Filter genome_amr_df to one drug + one evidence type, binarize resistant_phenotype,
    drop Intermediate/Non-susceptible and any conflicting duplicate genome_ids, return
    a DataFrame of (genome_id, label)."""
    filtered = genome_amr_df[(genome_amr_df['antibiotic'] == drug) & (genome_amr_df['evidence'] == evidence_type)]  # filtered = genome_amr_df[(genome_amr_df['antibiotic'] == drug) & (genome_amr_df['evidence'] == evidence_type)]
    filtered = filtered[filtered['resistant_phenotype'].isin(['Resistant', 'Susceptible'])]  # filtered = filtered[filtered['resistant_phenotype'].isin(['Resistant', 'Susceptible'])]
    dupes = filtered[filtered.duplicated(subset='genome_id', keep=False)]  # dupes = filtered[filtered.duplicated(subset='genome_id', keep=False)]
    print(f"{drug} ({evidence_type}): {dupes['genome_id'].nunique()} genomes with conflicting duplicate rows -- dropping them")  # dupes['genome_id'].nunique()
    filtered = filtered.drop_duplicates(subset='genome_id', keep=False).copy()  # filtered = filtered.drop_duplicates(subset='genome_id', keep=False).copy()
    filtered['label'] = filtered['resistant_phenotype'].map({'Resistant': 1, 'Susceptible': 0})  # filtered['resistant_phenotype'].map({'Resistant': 1, 'Susceptible': 0})
    return filtered[['genome_id', 'label']]  # filtered[['genome_id', 'label']]

meth_labels = build_labels(genome_amr_df, "methicillin")  # meth_labels = build_labels(genome_amr_df, "methicillin")
ery_labels = build_labels(genome_amr_df, "erythromycin")  # ery_labels = build_labels(genome_amr_df, "erythromycin")

# Set aside the Computational-Method-evidence rows (BV-BRC's own AdaBoost calls) for Step 4b's baseline
meth_genotype_calls = build_labels(genome_amr_df, "methicillin", evidence_type="Computational Method")  # meth_genotype_calls = build_labels(genome_amr_df, "methicillin", evidence_type="Computational Method")
ery_genotype_calls = build_labels(genome_amr_df, "erythromycin", evidence_type="Computational Method")  # ery_genotype_calls = build_labels(genome_amr_df, "erythromycin", evidence_type="Computational Method")

print("Methicillin labeled isolates:", len(meth_labels))     # len(meth_labels)
print("Erythromycin labeled isolates:", len(ery_labels))   # len(ery_labels)
print("Methicillin resistant fraction:", meth_labels['label'].mean())   # meth_labels['label'].mean()
print("Erythromycin resistant fraction:", ery_labels['label'].mean()) # ery_labels['label'].mean()


methicillin (Laboratory Method): 0 genomes with conflicting duplicate rows -- dropping them
erythromycin (Laboratory Method): 0 genomes with conflicting duplicate rows -- dropping them
methicillin (Computational Method): 0 genomes with conflicting duplicate rows -- dropping them
erythromycin (Computational Method): 0 genomes with conflicting duplicate rows -- dropping them
Methicillin labeled isolates: 221
Erythromycin labeled isolates: 441
Methicillin resistant fraction: 0.5158371040723982
Erythromycin resistant fraction: 0.3922902494331066


## Step 2 — Build a BROAD curated gene feature matrix  ⟵ WRITE IT

**This step is redesigned from the original K. pneumoniae plan — read this before writing code.**

**What changed and why:** the original design pre-filtered `sp_gene` to only the known causal gene family per drug (e.g. only KPC/NDM/... for meropenem). Measuring the real overlap for *K. pneumoniae* showed that collapsed the usable genome count to single digits — even for *S. aureus*, checking mecA/erm specifically (not just "any AMR gene") gave only 8 and 2 overlapping genomes. The fix: **don't pre-filter by gene family.** Build the feature matrix from EVERY curated AMR-property gene (keeping genes present in enough genomes to be learnable), and let the model discover which ones predict resistance. This is also just more realistic ML practice — you're not supposed to already know the answer before building the classifier.

**BIOLOGY (what we expect the model to find, not what we're assuming into the features):** methicillin resistance should be driven by the **mecA/mecC** cassette (real gene names seen live: `mecA`, `mecA_1`, `mecA_2`, `mecI`, `mecR1`). Erythromycin resistance should be driven by the **erm** gene family (real gene names seen live: `Erm(A)`, `Erm(B)`, `Erm(C)`, `ErmA`, `ErmC`, `ErmT`, `erm(A)`, `erm(C)`, `ermA`, `ermC`, and variants). Step 5 checks whether these actually come out on top.

**WHAT/WHY/HOW:**
1. Count how many distinct genomes carry each gene (`sp_gene_df.groupby('gene')['genome_id'].nunique()`).
2. Drop genes present in fewer than `MIN_GENOME_COUNT` genomes (a disclosed modeling choice — a gene seen in 1–2 genomes can't teach the model anything generalizable, and it just adds noise/dimensionality).
3. Pivot the remaining rows to an **isolate × gene** 0/1 matrix (one isolate can carry more than one relevant gene).

**Docs/functions to use:** `.groupby(...).nunique()`, boolean filtering with `.isin()`, `pandas.pivot_table` / `.pivot()` with `fill_value=0`.

In [6]:
# Inspect first -- uncomment and run:
print(sp_gene_df['gene'].value_counts().head(30))

MIN_GENOME_COUNT = 5  # drop AMR genes present in fewer than this many genomes -- too rare to learn from

# For reference only (NOT used to filter features) -- what we expect Step 5 to rediscover:
EXPECTED_GENE_MARKERS = {
    "methicillin": ["mecA", "mecA_1", "mecA_2", "mecI", "mecR1"],
    "erythromycin": ["Erm(A)", "Erm(B)", "Erm(C)", "ErmA", "ErmC", "ErmT", "erm(A)", "erm(C)", "ermA", "ermC"],
}

def build_feature_matrix(sp_gene_df, min_count=MIN_GENOME_COUNT):
    """Pivot ALL curated AMR-property genes (not filtered to any drug/family) to an
    isolate x gene 0/1 DataFrame, keeping only genes present in >= min_count genomes."""
    gene_counts = sp_gene_df.groupby('gene')['genome_id'].nunique() # gene_counts = sp_gene_df.groupby('gene')['genome_id'].nunique()
    keep_genes = gene_counts[gene_counts >= min_count].index   # keep_genes = gene_counts[gene_counts >= min_count].index
    filtered = sp_gene_df[sp_gene_df['gene'].isin(keep_genes)].copy()      # filtered = sp_gene_df[sp_gene_df['gene'].isin(keep_genes)].copy()
    filtered['present'] = 1
    feature_matrix = filtered.pivot_table(index='genome_id', columns='gene', values='present', aggfunc='max', fill_value=0)  # feature_matrix = filtered.pivot_table(index='genome_id', columns='gene', values='present', aggfunc='max', fill_value=0)
    return feature_matrix.reset_index()  # feature_matrix.reset_index()

amr_gene_features = build_feature_matrix(sp_gene_df)  # amr_gene_features = build_feature_matrix(sp_gene_df)

print("Shared AMR-gene feature matrix shape:", amr_gene_features.shape)  # amr_gene_features.shape
print("Number of gene features kept (>= MIN_GENOME_COUNT genomes):", amr_gene_features.shape[1] - 1)  # amr_gene_features.shape[1] - 1
# Sanity check: are the expected marker genes actually present as columns?
print("mecA-family columns present:", [c for c in amr_gene_features.columns if 'mec' in c.lower()])   # [c for c in amr_gene_features.columns if 'mec' in c.lower()]
print("erm-family columns present:", [c for c in amr_gene_features.columns if 'erm' in c.lower()])    # [c for c in amr_gene_features.columns if 'erm' in c.lower()]


gene
gyrA         126
gyrB         117
rpoB         104
rpoC          98
blaZ          96
GdpD          86
pgsA          81
mprF          80
rho           77
mgrA          76
MurA          73
arlS          69
dfrC          54
arlR          53
parE          53
Iso-tRNA      49
gidB          47
folA, Dfr     46
dfrA          46
mepR          45
norA          44
kasA          44
Alr           43
folP          41
mecR1         40
Ddl           40
mepA          40
sav1866       38
tetA          37
LiaF          37
Name: count, dtype: int64
Shared AMR-gene feature matrix shape: (2465, 120)
Number of gene features kept (>= MIN_GENOME_COUNT genomes): 119
mecA-family columns present: ['mecA', 'mecA_1', 'mecA_2', 'mecI', 'mecR1']
erm-family columns present: ['Erm(C)', 'ErmC', 'ermA', 'ermC']


## Step 2b — Assemble modeling frames + export bridge CSVs for the R twin (GIVEN pattern, blanks for the join)

**WHY bridge CSVs:** the R twin (`02_build.R`) models the SAME real, cleaned data rather than a separate live fetch — a deliberate cross-language bridge, same pattern used elsewhere in this portfolio.

**Note:** both drugs share the SAME `amr_gene_features` matrix built in Step 2 (it wasn't drug-specific) — the inner join with each drug's `label_df` naturally restricts each modeling frame to the genomes relevant to that drug.

In [7]:
def assemble_modeling_frame(features_df, labels_df, genome_meta_df):
    gene_cols = [c for c in features_df.columns if c != 'genome_id']
    merged = labels_df.merge(features_df, on='genome_id', how='left')
    merged[gene_cols] = merged[gene_cols].fillna(0)
    merged = merged.merge(genome_meta_df[['genome_id', 'mlst']], on='genome_id', how='inner')
    n_before = len(merged)
    merged = merged.dropna(subset=['mlst'])
    print(f"dropped {n_before - len(merged)} rows with missing mlst")
    return merged

meth_frame = assemble_modeling_frame(amr_gene_features, meth_labels, genome_meta_df)  # meth_frame = assemble_modeling_frame(amr_gene_features, meth_labels, genome_meta_df)
ery_frame = assemble_modeling_frame(amr_gene_features, ery_labels, genome_meta_df)  # ery_frame = assemble_modeling_frame(amr_gene_features, ery_labels, genome_meta_df)

print("Methicillin modeling frame:", meth_frame.shape)    # meth_frame.shape
print("Erythromycin modeling frame:", ery_frame.shape)  # ery_frame.shape

meth_frame.to_csv("data_py/meth_frame.csv", index=False)  # meth_frame.to_csv("data_py/meth_frame.csv", index=False)
ery_frame.to_csv("data_py/ery_frame.csv", index=False)  # ery_frame.to_csv("data_py/ery_frame.csv", index=False)
print("bridge CSVs exported for 02_build.R")


dropped 2 rows with missing mlst
dropped 2 rows with missing mlst
Methicillin modeling frame: (219, 122)
Erythromycin modeling frame: (439, 122)
bridge CSVs exported for 02_build.R


In [8]:
print("amr_gene_features total genomes (post MIN_GENOME_COUNT filter):", amr_gene_features.shape[0])
print("amr_gene_features total gene columns kept:", amr_gene_features.shape[1] - 1)

meth_ids = set(meth_labels['genome_id'])
feat_ids = set(amr_gene_features['genome_id'])
meta_ids = set(genome_meta_df['genome_id'])

print("\nmeth_labels genomes (both classes):", len(meth_ids))
print("meth_labels class balance:\n", meth_labels['label'].value_counts())

overlap_labels_features = meth_labels[meth_labels['genome_id'].isin(feat_ids)]
print("\noverlap meth_labels & amr_gene_features:", len(overlap_labels_features))
print("class balance at that overlap:\n", overlap_labels_features['label'].value_counts())

overlap_all = meth_labels[meth_labels['genome_id'].isin(feat_ids & meta_ids)]
print("\noverlap meth_labels & amr_gene_features & has mlst:", len(overlap_all))
print("class balance at that overlap:\n", overlap_all['label'].value_counts())

amr_gene_features total genomes (post MIN_GENOME_COUNT filter): 2465
amr_gene_features total gene columns kept: 119

meth_labels genomes (both classes): 221
meth_labels class balance:
 label
1    114
0    107
Name: count, dtype: int64

overlap meth_labels & amr_gene_features: 21
class balance at that overlap:
 label
1    21
Name: count, dtype: int64

overlap meth_labels & amr_gene_features & has mlst: 21
class balance at that overlap:
 label
1    21
Name: count, dtype: int64


## Step 3 — MLST-aware split vs. random split  ⟵ WRITE IT

**BIOLOGY:** MLST sequence types mark clonal lineages — e.g. ST5, ST8/USA300, ST22/EMRSA-15 are globally disseminated MRSA clones. Isolates within an ST share genetic background for reasons unrelated to any single resistance gene's mechanism — the same structure-leakage risk as TB's lineages or QSAR's scaffolds.

**WHAT/WHY/HOW:** Build TWO splits per drug: (a) **MLST-aware** — hold out whole sequence types; (b) plain **random** — for direct comparison. You'll compare their test performance in Step 4.

**Docs/functions to use:** `sklearn.model_selection.GroupShuffleSplit` (group = mlst) for the MLST-aware split; `sklearn.model_selection.train_test_split` (`stratify=` on the label) for the random split.

In [9]:
print("meth_frame shape:", meth_frame.shape)
print("distinct MLST groups:", meth_frame['mlst'].nunique())
print("label distribution:\n", meth_frame['label'].value_counts())

mlst_label_stats = meth_frame.groupby('mlst')['label'].agg(['count', 'mean'])
n_pure = ((mlst_label_stats['mean'] == 0) | (mlst_label_stats['mean'] == 1)).sum()
print(f"\n{n_pure} of {len(mlst_label_stats)} MLST groups are 'pure' (100% resistant or 100% susceptible)")
print("\nLargest groups:\n", mlst_label_stats.sort_values('count', ascending=False).head(15))

meth_frame shape: (219, 122)
distinct MLST groups: 40
label distribution:
 label
1    113
0    106
Name: count, dtype: int64

32 of 40 MLST groups are 'pure' (100% resistant or 100% susceptible)

Largest groups:
                   count      mean
mlst                             
MLST.saureus.22      79  0.873418
MLST.saureus.36      20  0.900000
MLST.saureus.45      15  0.066667
MLST.saureus.30      13  0.076923
MLST.saureus.5       12  0.333333
MLST.saureus.15      10  0.000000
MLST.saureus.8       10  0.800000
MLST.saureus.1        9  0.333333
MLST.saureus.59       7  0.142857
MLST.saureus.34       5  0.000000
MLST.saureus.12       3  0.000000
MLST.saureus.7        3  0.000000
MLST.saureus.72       2  0.000000
MLST.saureus.582      2  0.000000
MLST.saureus.25       2  0.000000


In [10]:
print(pd.crosstab(meth_frame['mecA'], meth_frame['label']))
print("\nmecA present fraction among resistant:", meth_frame.loc[meth_frame['label']==1, 'mecA'].mean())
print("mecA present fraction among susceptible:", meth_frame.loc[meth_frame['label']==0, 'mecA'].mean())

erm_cols = [c for c in ery_frame.columns if 'erm' in c.lower()]
print("\nerm-family columns:", erm_cols)
for c in erm_cols:
    print(f"{c} present fraction among resistant:", ery_frame.loc[ery_frame['label']==1, c].mean(),
          "| among susceptible:", ery_frame.loc[ery_frame['label']==0, c].mean())

label    0    1
mecA           
0.0    106  113

mecA present fraction among resistant: 0.0
mecA present fraction among susceptible: 0.0

erm-family columns: ['Erm(C)', 'ErmC', 'ermA', 'ermC']
Erm(C) present fraction among resistant: 0.0 | among susceptible: 0.0
ErmC present fraction among resistant: 0.005780346820809248 | among susceptible: 0.0
ermA present fraction among resistant: 0.0 | among susceptible: 0.0
ermC present fraction among resistant: 0.0 | among susceptible: 0.0


In [11]:
mec_cols = [c for c in meth_frame.columns if 'mec' in c.lower()]
print("mec-family columns in meth_frame:", mec_cols)
for c in mec_cols:
    print(f"{c}: present fraction among resistant = {meth_frame.loc[meth_frame['label']==1, c].mean():.3f}"
          f" | among susceptible = {meth_frame.loc[meth_frame['label']==0, c].mean():.3f}")

mec-family columns in meth_frame: ['mecA', 'mecA_1', 'mecA_2', 'mecI', 'mecR1']
mecA: present fraction among resistant = 0.000 | among susceptible = 0.000
mecA_1: present fraction among resistant = 0.000 | among susceptible = 0.000
mecA_2: present fraction among resistant = 0.000 | among susceptible = 0.000
mecI: present fraction among resistant = 0.000 | among susceptible = 0.000
mecR1: present fraction among resistant = 0.000 | among susceptible = 0.000


In [12]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split

def mlst_aware_split(frame, feature_cols, test_size=0.2, random_state=0, max_attempts=20):
    X = frame[feature_cols]
    y = frame['label']
    groups = frame['mlst']
    for attempt in range(max_attempts):
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state + attempt)
        train_idx, test_idx = next(splitter.split(X, y, groups))
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        if y_train.nunique() == 2 and y_test.nunique() == 2:
            return X.iloc[train_idx], X.iloc[test_idx], y_train, y_test
    raise ValueError(
        f"Could not find an MLST-aware split with both classes on both sides after {max_attempts} attempts -- "
        "this itself is worth reporting: it means this drug's resistant/susceptible isolates are concentrated "
        "in relatively few MLST groups."
    )

def random_split(frame, feature_cols, test_size=0.2, random_state=0):
    """Return X_train, X_test, y_train, y_test from a naive random (label-stratified) split."""
    X = frame[feature_cols]  # X = frame[feature_cols]
    y = frame['label']  # y = frame['label']
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)  # train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)

meth_feature_cols = [c for c in meth_frame.columns if c not in ('genome_id', 'label', 'mlst')]  # [c for c in meth_frame.columns if c not in ('genome_id', 'label', 'mlst')]
ery_feature_cols = [c for c in ery_frame.columns if c not in ('genome_id', 'label', 'mlst')]   # [c for c in ery_frame.columns if c not in ('genome_id', 'label', 'mlst')]

meth_mlst_split = mlst_aware_split(meth_frame, meth_feature_cols)    # mlst_aware_split(meth_frame, meth_feature_cols)
meth_random_split = random_split(meth_frame, meth_feature_cols)  # random_split(meth_frame, meth_feature_cols)
ery_mlst_split = mlst_aware_split(ery_frame, ery_feature_cols)     # mlst_aware_split(ery_frame, ery_feature_cols)
ery_random_split = random_split(ery_frame, ery_feature_cols)   # random_split(ery_frame, ery_feature_cols)


## Step 4 — Train + evaluate per drug, on BOTH splits  ⟵ WRITE IT

**BIOLOGY/WHY:** A false negative (resistant strain called susceptible) risks treatment failure and continued spread — report sensitivity and specificity separately.

**WHAT/WHY/HOW:** Fit **Logistic Regression** and **Random Forest** per drug, per split. Report sensitivity, specificity, AUROC.

**Docs/functions to use:** `sklearn.linear_model.LogisticRegression`, `sklearn.ensemble.RandomForestClassifier`, `sklearn.metrics.confusion_matrix`, `sklearn.metrics.roc_auc_score`.

In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score

def evaluate_model(model, X_train, X_test, y_train, y_test):
    """Fit `model` on train, return a dict of sensitivity, specificity, auroc on test."""
    model.fit(X_train, y_train)  # model.fit(X_train, y_train)
    y_pred = model.predict(X_test)  # model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]  # model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()  # confusion_matrix(y_test, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float('nan') # tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')  # tn / (tn + fp) if (tn + fp) > 0 else float('nan')
    auroc = roc_auc_score(y_test, y_proba)  # roc_auc_score(y_test, y_proba)
    return {'sensitivity': sensitivity, 'specificity': specificity, 'auroc': auroc}

results = []

drug_splits = {
    "methicillin": {"mlst-aware": meth_mlst_split, "random": meth_random_split},
    "erythromycin": {"mlst-aware": ery_mlst_split, "random": ery_random_split},
}
model_factories = {
    "LogReg": lambda: LogisticRegression(max_iter=1000),
    "RandomForest": lambda: RandomForestClassifier(random_state=0),
}

for drug, splits in drug_splits.items():      # drug_splits.items()
    for split_name, split in splits.items():      # splits.items()
        X_train, X_test, y_train, y_test = split
        for model_name, make_model in model_factories.items():      # model_factories.items()
            model = make_model()     # make_model()
            metrics = evaluate_model(model, X_train, X_test, y_train, y_test)      # evaluate_model(model, X_train, X_test, y_train, y_test)
            results.append({"drug": drug, "split": split_name, "model": model_name, **metrics})

results_df = pd.DataFrame(results)  # results_df = pd.DataFrame(results)
print(results_df)


           drug       split         model  sensitivity  specificity     auroc
0   methicillin  mlst-aware        LogReg     1.000000     0.000000  0.500000
1   methicillin  mlst-aware  RandomForest     1.000000     0.000000  0.500000
2   methicillin      random        LogReg     1.000000     0.000000  0.586957
3   methicillin      random  RandomForest     0.173913     1.000000  0.586957
4  erythromycin  mlst-aware        LogReg     0.000000     0.985294  0.472269
5  erythromycin  mlst-aware  RandomForest     0.000000     0.970588  0.485714
6  erythromycin      random        LogReg     0.000000     0.962264  0.444744
7  erythromycin      random  RandomForest     0.000000     0.962264  0.444744


## Step 4b — Compare against BV-BRC's own genotype-based calls  ⟵ WRITE IT

**WHY:** the `evidence == "Computational Method"` rows you set aside in Step 1 (`meth_genotype_calls`/`ery_genotype_calls`) are BV-BRC's own AdaBoost-classifier resistance calls — a ready-made baseline, each carrying a self-reported `computational_method_performance` string. If present for enough overlapping isolates, join them to your MLST-aware test-set isolates and compute their sensitivity/specificity against the same real `Laboratory Method`-evidence ground truth, then add as a baseline row to `results_df`.

In [14]:
for drug, labels, genotype_calls in [
    ("methicillin", meth_labels, meth_genotype_calls),
    ("erythromycin", ery_labels, ery_genotype_calls),
]:
    overlap = labels.merge(genotype_calls, on='genome_id', suffixes=('_true', '_computed'))  # overlap = labels.merge(genotype_calls, on='genome_id', suffixes=('_true', '_computed'))
    print(f"{drug}: {len(overlap)} genomes with both a Laboratory Method label and a Computational Method call")  # len(overlap)
    if len(overlap) > 0:
        tn, fp, fn, tp = confusion_matrix(overlap['label_true'], overlap['label_computed']).ravel()  # confusion_matrix(overlap['label_true'], overlap['label_computed']).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else float('nan')  # tp / (tp + fn) if (tp + fn) > 0 else float('nan')
        specificity = tn / (tn + fp) if (tn + fp) > 0 else float('nan')  # tn / (tn + fp) if (tn + fp) > 0 else float('nan')
        results.append({
            "drug": drug, "split": "n/a (BV-BRC baseline)", "model": "BV-BRC AdaBoost",
            "sensitivity": sensitivity, "specificity": specificity, "auroc": float('nan'),
        })

results_df = pd.DataFrame(results)   # results_df = pd.DataFrame(results)  -- rebuild after appending the baseline rows


methicillin: 11 genomes with both a Laboratory Method label and a Computational Method call
erythromycin: 32 genomes with both a Laboratory Method label and a Computational Method call


## Step 5 — Feature importance: does the model rediscover mecA/erm on its own?  ⟵ WRITE IT (HERO)

**This is the real payoff of Step 2's redesign.** Nothing in the feature matrix told the model which genes matter — `amr_gene_features` includes every AMR gene seen in >= `MIN_GENOME_COUNT` genomes, aminoglycoside genes, efflux pumps, everything. If methicillin's top-ranked features are `mecA`/`mecA_1`/`mecA_2`/`mecI`/`mecR1`, and erythromycin's top-ranked features are from the `erm` family, that's the model independently rediscovering real, published AMR biology — a much stronger check than Step 2 in the original K. pneumoniae design, where the genes were already hand-picked before training.

**Docs/functions to use:** `.coef_[0]` (LogisticRegression), `.feature_importances_` (RandomForestClassifier), paired with the feature column names, sorted descending.

In [15]:
def get_feature_importance(fitted_model, feature_names):
    """Return a pandas Series of feature -> importance/coefficient, sorted descending by magnitude."""
    if hasattr(fitted_model, 'coef_'):
        importances = importances = fitted_model.coef_[0]  # importances = fitted_model.coef_[0]
    elif hasattr(fitted_model, 'feature_importances_'):
        importances = fitted_model.feature_importances_  # importances = fitted_model.feature_importances_
    else:
        raise ValueError(f"unsupported model type: {type(fitted_model)}")
    return pd.Series(importances, index=feature_names).sort_values(key=abs, ascending=False)  # pd.Series(importances, index=feature_names).sort_values(key=abs, ascending=False)

feature_importance_records = []

for drug, splits in drug_splits.items():      # drug_splits.items()
    X_train, X_test, y_train, y_test = splits["mlst-aware"]
    for model_name, make_model in model_factories.items():      # model_factories.items()
        model = make_model()     # make_model()
        model.fit(X_train, y_train)      # model.fit(X_train, y_train)
        importances = get_feature_importance(model, X_train.columns)      # get_feature_importance(model, X_train.columns)
        print(f"\n{drug} / {model_name} top 10 features (mlst-aware split):\n", importances.head(10))
        for feat, val in importances.items():      # importances.items()
            feature_importance_records.append({"drug": drug, "model": model_name, "feature": feat, "importance": val})

print("\nCompare the top features above against EXPECTED_GENE_MARKERS from Step 2 -- did mecA/erm-family genes actually rise to the top?")



methicillin / LogReg top 10 features (mlst-aware split):
 S10p           0.696160
gyrB           0.590979
gyrA           0.590979
blaZ           0.523357
rpoC           0.523357
S12p           0.523357
BlaZ family    0.523357
BceR           0.500900
TcaB2          0.340246
GdpD           0.340246
dtype: float64

methicillin / RandomForest top 10 features (mlst-aware split):
 S10p           0.098597
gyrB           0.094541
gyrA           0.094242
blaZ           0.085130
rpoC           0.071946
S12p           0.068790
BceR           0.065647
BlaZ family    0.064668
ANT(4')-Ib     0.054056
rpoB           0.051481
dtype: float64

erythromycin / LogReg top 10 features (mlst-aware split):
 EF-G      0.528761
fabF      0.528761
MurA      0.528761
mgrA     -0.490471
arlR     -0.471386
graR     -0.461434
norB_3   -0.461434
rho       0.439679
PgsA      0.439679
Ddl       0.439679
dtype: float64

erythromycin / RandomForest top 10 features (mlst-aware split):
 MurA         0.076231
EF-G         

## Step 6 — Export results (GIVEN pattern, fill in the paths)

In [16]:
os.makedirs("results_py", exist_ok=True)

results_df.to_csv("results_py/model_performance.csv", index=False)  # results_df.to_csv("results_py/model_performance.csv", index=False)
pd.DataFrame(feature_importance_records).to_csv("results_py/feature_importance.csv", index=False)  # pd.DataFrame(feature_importance_records).to_csv("results_py/feature_importance.csv", index=False)

print("Done. Now write INTERPRETATION.md and INTERVIEW_QA.md from these real results,")
print("and fill in PROJECT_NARRATIVE.md sections 8-9 with what actually happened.")


Done. Now write INTERPRETATION.md and INTERVIEW_QA.md from these real results,
and fill in PROJECT_NARRATIVE.md sections 8-9 with what actually happened.
